# HeartShare harmonization — BDC runtime notebook (updated)

This is the primary, thin BDC notebook. It loads the versioned release
manifest, verifies the declared HeartShare runtime archive, installs bundled
offline wheels when needed, and runs the shared harmonization package.

**Quick start — edit up to four cells, then run top to bottom:**

| Step | Action | What you set |
|---|---|---|
| **1. Paths** | Edit if needed | Release location/version and optional output folder |
| **2. Available studies & variables** | Run as-is | Lists valid study names, variable names, labels, and units |
| **3. Select studies** | Edit | Which studies to harmonize (`[]` = all studies) |
| **4. Select domains & variables** | Edit | Which harmonized variables/domains to include |
| **5. Run harmonization** | Edit controls | Toggle `DRY_RUN` (`True` first, then `False`) and `BASELINE_ONLY` |
| **6. Review output** | Run as-is | Summarizes coverage, missingness, and prints a Table One handoff snippet |

**What you get** in the run folder: `harmonized_long.csv` (the deliverable) and
`harmonized_long.parquet`, plus `harmonized_lineage.parquet`, `manifest.json`,
`missing_variables.json/.log`, `variable_dictionary.json/.csv`, `pull.log`, and
`pipeline.log`.

**Output shape** — one row per participant × variable × source file ×
measurement context (long format). A missing measurement produces **no row**,
so a lower-than-expected row count is usually correct.

**Runtime** — seconds for a dry run, a few minutes for a full multi-study pull.


## 1. Paths (edit if needed)


In [ ]:
# ---------------------------------------------------------------------------
# 1. PATHS — edit common settings below if your project uses different paths.
# ---------------------------------------------------------------------------
from pathlib import Path

# === Common settings =======================================================
# Folder containing the versioned release directories (catalog + standards + mappings).
RELEASE_ROOT = "/sbgenomics/project-files/x01_harmonization"
RELEASE_VERSION = "v0.1.6"          # or "latest" to select the newest release

# Where results go. None -> /sbgenomics/workspace/output-files/harmonized_<date>_<run_id>,
# which syncs back to the Seven Bridges project automatically.
OUTPUT_DIR = None

# Optional label for this run (e.g. "topcat_pilot").
# Leave as None to auto-generate a fresh UTC timestamp each time Step 5 runs.
RUN_ID = None

# === Advanced settings (usually leave as default) ==========================
# Staged folder override only when a study's catalog path is unavailable.
# Changes source-file resolution, not the STUDIES selection.
STUDY_DATA_DIRS = []

# Read only the first N participants per file (useful for a quick smoke test).
LIMIT = None

# "warn" records unexpected categorical codes and continues.
# "strict" fails the affected dataset instead.
DRIFT_POLICY = "warn"

# Notebook shows warnings and errors; detailed INFO messages go to pipeline.log.
LOG_LEVEL = "WARNING"

print(f"Release : {RELEASE_ROOT} ({RELEASE_VERSION})")
if STUDY_DATA_DIRS:
    print(f"Sources : {len(STUDY_DATA_DIRS)} staged folder override(s)")
else:
    print("Sources : release catalog paths (no staged folder overrides)")
print(f"Run ID  : {RUN_ID or '(auto-generated in Step 5)'}")
print("Mode    : choose DRY_RUN and BASELINE_ONLY in Step 5")


### Runtime setup (do not edit — run as-is)


In [ ]:
"""Small standard-library bootstrap used by HeartShare thin BDC notebooks."""

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from zipfile import ZipFile


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def runtime_spec_from_release(manifest_path: Path) -> dict[str, object]:
    try:
        import yaml
    except ImportError as exc:
        raise RuntimeError("PyYAML is required to read the release manifest") from exc
    document = yaml.safe_load(Path(manifest_path).read_text(encoding="utf-8")) or {}
    retired = document.get("retired")
    if retired and os.environ.get("HEARTSHARE_ALLOW_RETIRED_RELEASE") != "1":
        raise RuntimeError(
            f"Release {document.get('release_version')} is RETIRED: {retired}. Use the release that "
            "supersedes it (set HEARTSHARE_ALLOW_RETIRED_RELEASE=1 only to reproduce an old result)."
        )
    spec = document.get("runtime_bundle")
    if not isinstance(spec, dict) or not spec.get("path") or not spec.get("sha256"):
        raise ValueError(
            f"Release manifest has no complete runtime_bundle path/checksum: {manifest_path}"
        )
    resolved = dict(spec)
    path = Path(str(spec["path"]))
    if not path.is_absolute():
        path = (Path(manifest_path).parent / path).resolve()
    resolved["path"] = path
    # What this release's mapping payload needs from its runtime (written by the release manager).
    resolved["required_runtime_features"] = list(document.get("required_runtime_features") or [])
    return resolved


def _safe_extract(archive_path: Path, target: Path) -> None:
    with ZipFile(archive_path) as archive:
        root = target.resolve()
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if destination != root and root not in destination.parents:
                raise ValueError(f"Unsafe path in runtime archive: {member.filename}")
        archive.extractall(target)


def _verify_extracted(target: Path) -> dict[str, object]:
    manifest_path = target / "runtime_manifest.json"
    if not manifest_path.exists():
        raise ValueError("Runtime archive is missing runtime_manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for name, expected in (manifest.get("files") or {}).items():
        path = target / name
        if not path.is_file():
            raise ValueError(f"Runtime archive is missing declared file: {name}")
        actual = sha256_file(path)
        if actual != expected:
            raise ValueError(
                f"Runtime file checksum mismatch for {name}: expected {expected}, got {actual}"
            )
    return manifest


def _install_offline_wheels(target: Path, manifest: dict[str, object]) -> Path | None:
    requirements = list(manifest.get("required_offline_distributions") or [])
    if not requirements:
        return None
    wheelhouse = target / "wheelhouse"
    vendor = target / "vendor"
    marker = vendor / ".heartshare_wheels_installed"
    if marker.exists():
        return vendor
    vendor.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-index",
        "--find-links",
        str(wheelhouse),
        "--target",
        str(vendor),
        *requirements,
    ]
    subprocess.run(command, check=True)
    marker.write_text("ok\n", encoding="utf-8")
    return vendor


def bootstrap_runtime(
    archive_path: Path,
    expected_sha256: str,
    *,
    workspace_root: Path | None = None,
    expected_output_schema: str = "1.3",
    required_runtime_features: list[str] | tuple[str, ...] = (),
    supported_output_schemas: tuple[str, ...] | None = None,
) -> tuple[Path, dict[str, object]]:
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        raise FileNotFoundError(f"Runtime bundle not found: {archive_path}")
    actual_sha256 = sha256_file(archive_path)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            "Runtime bundle checksum mismatch: "
            f"expected {expected_sha256}, got {actual_sha256}"
        )

    workspace_root = Path(
        workspace_root
        or os.environ.get(
            "HEARTSHARE_VENDOR_ROOT", "/sbgenomics/workspace/_heartshare_vendor"
        )
    )
    workspace_root.mkdir(parents=True, exist_ok=True)
    target = workspace_root / actual_sha256[:16]
    if not (target / "runtime_manifest.json").exists():
        with tempfile.TemporaryDirectory(
            prefix="heartshare-runtime-", dir=workspace_root
        ) as temporary:
            staging = Path(temporary) / "unpacked"
            staging.mkdir()
            _safe_extract(archive_path, staging)
            _verify_extracted(staging)
            try:
                staging.rename(target)
            except FileExistsError:
                shutil.rmtree(staging)

    manifest = _verify_extracted(target)
    supported = supported_output_schemas or (expected_output_schema,)
    if str(manifest.get("output_schema_version")) not in supported:
        raise ValueError(
            "Runtime/output schema mismatch: "
            f"expected one of {supported}, runtime provides "
            f"{manifest.get('output_schema_version')}"
        )

    unsupported = sorted(set(required_runtime_features) - set(manifest.get("runtime_features") or ()))
    if unsupported:
        raise ValueError(
            "This release's mappings need runtime features the bundled runtime "
            f"({manifest.get('runtime_version')}) does not provide: {unsupported}. The release is "
            "mis-assembled — do not run it; rebuild it with a current runtime bundle."
        )

    vendor = _install_offline_wheels(target, manifest)
    import_root = target / str(manifest.get("import_root") or "bdc/src")
    for path in [vendor, target, import_root]:
        if path is not None and str(path) not in sys.path:
            sys.path.insert(0, str(path))

    os.environ["HEARTSHARE_RUNTIME_BUNDLE_PATH"] = str(archive_path)
    os.environ["HEARTSHARE_RUNTIME_BUNDLE_SHA256"] = actual_sha256
    os.environ["HEARTSHARE_RUNTIME_VERSION"] = str(
        manifest.get("runtime_version") or ""
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_SHA"] = str(
        manifest.get("build_git_sha") or ""
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_BRANCH"] = str(
        manifest.get("build_git_branch") or "runtime-bundle"
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_CLEAN"] = str(
        bool(manifest.get("build_git_clean", True))
    ).lower()
    return import_root, manifest


# Resolve the release manifest without importing the runtime it declares.
_release_root = Path(RELEASE_ROOT)
_release_version = str(RELEASE_VERSION)
if _release_version == "latest":
    import re
    _versions = []
    for _child in _release_root.iterdir():
        _match = re.fullmatch(r"v(\d+)\.(\d+)\.(\d+)", _child.name)
        if _child.is_dir() and _match:
            _versions.append((tuple(int(_part) for _part in _match.groups()), _child.name))
    if not _versions:
        raise FileNotFoundError(f"No versioned release folders under {_release_root}")
    _release_version = sorted(_versions)[-1][1]
_release_manifest_path = (
    _release_root / _release_version / f"release_manifest_{_release_version}.yaml"
)
_runtime_spec = runtime_spec_from_release(_release_manifest_path)
_runtime_root, _runtime_manifest = bootstrap_runtime(
    Path(_runtime_spec["path"]),
    str(_runtime_spec["sha256"]),
    supported_output_schemas=("1.1", "1.2", "1.3"),
    required_runtime_features=_runtime_spec.get("required_runtime_features") or (),
)
from heartshare_harmonization import pull_and_harmonize as pipeline
print(
    f"Runtime {_runtime_manifest['runtime_version']} ready from "
    f"{_runtime_spec['path']}"
)


## 2. Available studies and variables (run as-is)


In [ ]:
# ---------------------------------------------------------------------------
# 2. AVAILABLE STUDIES & VARIABLES — read-only. Run as-is to see valid names.
# ---------------------------------------------------------------------------
from collections import defaultdict

from heartshare_harmonization.catalog import load_catalog
from heartshare_harmonization.pull_and_harmonize import load_standards_library
from heartshare_harmonization.release import (
    load_release_manifest,
    resolve_release_manifest_path,
)

_manifest_path = resolve_release_manifest_path(Path(RELEASE_ROOT), RELEASE_VERSION)
_release = load_release_manifest(_manifest_path)
_catalog = load_catalog(_release.catalog)
_standards = load_standards_library(_release.standards_library)

print(f"Release {_release.release_version}  ({_manifest_path})\n")

print("STUDIES — copy any of these into STUDIES in Step 3")
print("-" * 72)
for _name in _catalog.study_names():
    _study = _catalog.study(_name)
    _n = len(_study.datasets)
    _mapped = "mapped" if _study.hdo_mapping_file else "NO MAPPING FILE"
    print(f'  "{_name}"{" " * max(1, 22 - len(_name))} {_n:>2} file(s)   {_mapped}')

_by_category = defaultdict(list)
for _key, _definition in _standards.items():
    _by_category[_definition.get("category") or "other"].append((_key, _definition))

print("\nVARIABLES — copy variable keys into VARIABLES (or [domain] names into DOMAINS) in Step 4")
print("-" * 72)
for _category in sorted(_by_category):
    print(f"\n  [{_category}]")
    for _key, _defn in sorted(_by_category[_category], key=lambda item: item[0]):
        _label = _defn.get("display_name") or ""
        _units = _defn.get("standard_units") or _defn.get("units") or ""
        _unit_suffix = f" ({_units})" if _units else ""
        print(f'    "{_key}"{" " * max(1, 26 - len(_key))} {_label}{_unit_suffix}'.rstrip())

print(f"\n{len(_standards)} variables across {len(_by_category)} domains.")


## 3. Select studies (edit)


In [ ]:
# ---------------------------------------------------------------------------
# 3. SELECT STUDIES — edit this list.
#    Leave STUDIES = [] to select EVERY study in the release catalog.
# ---------------------------------------------------------------------------
STUDIES = [
    "HFN-NEAT",
    # "TOPCAT",
]

_available_studies = _catalog.study_names()
_selected_studies_preview = list(STUDIES) if STUDIES else list(_available_studies)
_unknown_studies = sorted(set(_selected_studies_preview) - set(_available_studies))
if _unknown_studies:
    raise ValueError(
        f"Study name(s) not present in release {_release.release_version}: "
        f"{_unknown_studies}. Valid studies: {_available_studies}"
    )

print(f"Selected {len(_selected_studies_preview)} study/studies: {_selected_studies_preview}")


## 4. Select domains and variables (edit)


In [ ]:
# ---------------------------------------------------------------------------
# 4. SELECT DOMAINS & VARIABLES — edit these lists.
#
#    DOMAINS selects all variables in those domains.
#    VARIABLES adds individual variables; both lists are combined without duplicates.
#    Leave both empty ([]) to harmonize EVERY mapped variable.
# ---------------------------------------------------------------------------
VARIABLES = [
    "age",
    "sex",
    # Include height and weight when you also want BMI derived across source files.
    "height",
    "weight",
    "bmi",
]

DOMAINS = [
    # "demographics",
    # "anthropometric",
    # "vitals",
]

if VARIABLES:
    print(f"Selecting {len(VARIABLES)} variable(s) by name: {VARIABLES}")
if DOMAINS:
    print(f"Adding every variable in domain(s): {DOMAINS}")
if not VARIABLES and not DOMAINS:
    print("Selecting ALL mapped variables.")
else:
    from argparse import Namespace
    _selection_preview = pipeline.resolve_requested_standards(
        Namespace(variables=VARIABLES, domains=DOMAINS), _standards
    )
    print(f"Combined selection ({len(_selection_preview)}): {_selection_preview}")


## 5. Run harmonization (edit DRY_RUN / BASELINE_ONLY)


In [ ]:
# ---------------------------------------------------------------------------
# 5. RUN HARMONIZATION — edit DRY_RUN / BASELINE_ONLY, then run this cell.
#    You do not need to return to Step 1 when switching DRY_RUN from True to False.
# ---------------------------------------------------------------------------
from datetime import datetime, timezone
import sys

# Start with DRY_RUN = True to verify file paths and mappings.
# When Step 6 reports READY, change this to False and re-run Steps 5 and 6.
DRY_RUN = True

# True keeps only mapping-designated baseline visits (including documented
# compatibility aliases such as a_base). False includes all visits.
BASELINE_ONLY = True

# Advanced: repeated dependency values are never silently averaged.
# Add an explicitly reviewed rule only when needed, e.g. "TOPCAT:weight:median".
DERIVATION_REDUCERS = []

_selected_studies = list(STUDIES) if STUDIES else _catalog.study_names()
_unknown_studies = sorted(set(_selected_studies) - set(_catalog.study_names()))
if _unknown_studies:
    raise ValueError(
        f"Study name(s) not present in release {_release.release_version}: "
        f"{_unknown_studies}"
    )

# Auto-generate a fresh UTC timestamp when RUN_ID is None, and keep dry-run
# outputs in a separate _dryrun folder when OUTPUT_DIR is not explicitly pinned.
_base_run_id = str(RUN_ID) if RUN_ID else datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EFFECTIVE_RUN_ID = f"{_base_run_id}_dryrun" if (DRY_RUN and not OUTPUT_DIR) else _base_run_id

argv = [
    "heartshare-harmonize",
    "--release-root", str(RELEASE_ROOT),
    "--release-version", str(RELEASE_VERSION),
    "--run-id", EFFECTIVE_RUN_ID,
    "--categorical-drift-policy", str(DRIFT_POLICY),
    "--log-level", str(LOG_LEVEL),
    "--emit-csv",
    "--skip-missing-mappings",
    "--no-emit-hdos",
]

if OUTPUT_DIR:
    argv += ["--output-dir", str(OUTPUT_DIR)]
if STUDY_DATA_DIRS:
    argv += ["--study-data-dirs", *[str(p) for p in STUDY_DATA_DIRS]]
argv += ["--studies", *_selected_studies]
if VARIABLES:
    argv += ["--variables", *VARIABLES]
if DOMAINS:
    argv += ["--domains", *DOMAINS]
if LIMIT:
    argv += ["--limit", str(LIMIT)]
if BASELINE_ONLY:
    argv += ["--baseline-only"]
if DERIVATION_REDUCERS:
    argv += ["--derivation-reducers", *DERIVATION_REDUCERS]
if DRY_RUN:
    argv += ["--dry-run"]

RUN_DIR = pipeline._resolve_output_dir(
    Path(OUTPUT_DIR) if OUTPUT_DIR else None, EFFECTIVE_RUN_ID, None
)

print(f"Mode          : {'DRY RUN (no data read)' if DRY_RUN else 'REAL RUN'}")
print(f"Baseline only : {BASELINE_ONLY}")
print(f"Studies ({len(_selected_studies)})  : {_selected_studies}")
print(f"Run ID        : {EFFECTIVE_RUN_ID}")
print(f"Output folder : {RUN_DIR}")
print("Command       : " + " ".join(argv[1:]))
print("-" * 70)

_saved_argv = sys.argv
try:
    sys.argv = argv
    EXIT_CODE = pipeline.main()
finally:
    sys.argv = _saved_argv

print("-" * 70)
if EXIT_CODE == 0:
    print("OK. Wrote:")
    for _f in sorted(Path(RUN_DIR).iterdir()):
        print(f"  {_f.name:<30} {_f.stat().st_size:>10,} bytes")
else:
    print(f"FAILED (exit code {EXIT_CODE}). Read the log above, then check")
    print(f"  {Path(RUN_DIR) / 'manifest.json'}  ->  the 'error' key.")
    raise RuntimeError(
        f"Harmonization failed with exit code {EXIT_CODE}; output inspection was stopped."
    )


## 6. Review output (run as-is)


In [ ]:
# ---------------------------------------------------------------------------
# 6. REVIEW OUTPUT — read-only summary of what was planned or produced.
# ---------------------------------------------------------------------------
import json

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain python, e.g. automated tests
    display = print

if DRY_RUN:
    _plan = json.loads((Path(RUN_DIR) / "dry_run_plan.json").read_text())
    manifest = json.loads((Path(RUN_DIR) / "manifest.json").read_text())
    _plan_frame = pd.DataFrame(_plan)
    _display_columns = [
        "study", "dataset_id", "exam_label", "status",
        "mount_exists", "mapping_file_exists", "notes",
    ]
    _coverage = pd.DataFrame(manifest.get("variable_coverage") or [])
    _attention = (
        _coverage[~_coverage["status"].isin(["planned", "produced"])].copy()
        if not _coverage.empty else _coverage
    )
    if _plan_frame.empty:
        print("NOT READY — the selected studies produced no datasets.")
    else:
        _source_files_found = int(_plan_frame["mount_exists"].sum())
        _mapping_files_found = int(_plan_frame["mapping_file_exists"].sum())
        print(
            f"DRY RUN — {_plan_frame['study'].nunique()} study/studies selected\n"
            f"{len(_plan_frame)} datasets planned\n"
            f"{_source_files_found} source files found\n"
            f"{_mapping_files_found} dataset mapping checks passed\n"
            "No data was read."
        )

    _not_ready = _plan_frame[
        (_plan_frame["status"] != "planned")
        | (~_plan_frame["mount_exists"])
        | (~_plan_frame["mapping_file_exists"])
    ] if not _plan_frame.empty else _plan_frame
    if _plan_frame.empty:
        pass  # The empty-plan warning above must not be followed by READY.
    elif not _not_ready.empty:
        print(f"\nNOT READY — {len(_not_ready)} dataset(s) need attention.")
        display(_not_ready[_display_columns])
        print("Fix the paths or mappings shown above before setting DRY_RUN = False.")
    elif _coverage.empty:
        print("\nFILES READY — variable coverage has not been assessed.")
        print("Use a runtime that reports variable coverage to check requested mappings.")
    elif not _attention.empty:
        print("\nREADY TO RUN, WITH VARIABLE COVERAGE GAPS")
        print("Every planned source file and mapping file exists, but some requested "
              "variables have coverage gaps.")
        print("Review the gaps below, then set DRY_RUN = False in Step 5 and re-run Steps 5 and 6.")
    else:
        print("\nREADY — every planned source file and mapping file exists; "
              "all assessed variables have planned mappings.")
        print("Set DRY_RUN = False in Step 5 and re-run Steps 5 and 6.")

    if not _attention.empty:
        print(f"\nVARIABLE COVERAGE GAPS — {len(_attention)} study/variable/timepoint(s)")
        _attention["timepoint"] = _attention["timepoint"].fillna("Study-level / unspecified")
        _gap_summary = (
            _attention.groupby(["standard_name", "timepoint", "status"], dropna=False)
            ["study"].agg(lambda values: ", ".join(sorted(set(values))))
            .rename("studies").reset_index()
        )
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(_gap_summary)
            display(_attention[[
                "study", "standard_name", "timepoint", "status", "detail"
            ]].sort_values(["standard_name", "timepoint", "study"]))

    if not _coverage.empty:
        print("\nPLANNED VARIABLE COVERAGE — variable/timepoint by study")
        _matrix = (
            _coverage.assign(timepoint=_coverage["timepoint"].fillna("Study-level / unspecified"))
            .pivot(index=["standard_name", "timepoint"], columns="study", values="status")
            .fillna("Not assessed at this timepoint")
        )
        with pd.option_context("display.max_rows", None, "display.max_columns", None):
            display(_matrix)
        print("planned = mapping found; values have not been checked. "
              "mapping_missing = no selected mapping, not proof the study never collected it.")
        print("Not assessed at this timepoint = no coverage record for that combination.")
        print("Source columns and usable values are checked during the real run.")
        print(f"Full coverage: {Path(RUN_DIR) / 'variable_coverage.csv'}")
else:
    _parquet_path = Path(RUN_DIR) / "harmonized_long.parquet"
    if _parquet_path.is_file():
        long_df = pd.read_parquet(_parquet_path)
    else:
        long_df = pd.read_csv(Path(RUN_DIR) / "harmonized_long.csv", low_memory=False)
    manifest = json.loads((Path(RUN_DIR) / "manifest.json").read_text())

    print(f"{len(long_df):,} rows x {len(long_df.columns)} columns")
    print(f"{long_df['participant_id'].nunique():,} participants, "
          f"{long_df['standard_name'].nunique()} variables, "
          f"{long_df['study'].nunique()} studies\n")

    print("BY STUDY")
    display(long_df.groupby("study").agg(
        rows=("standard_name", "size"),
        participants=("participant_id", "nunique"),
        variables=("standard_name", "nunique"),
    ))

    print("\nROWS BY VARIABLE AND STUDY")
    display(pd.crosstab(long_df["standard_name"], long_df["study"], margins=True))

    if "timepoint" in long_df.columns:
        _timepoints = sorted(long_df["timepoint"].dropna().astype(str).unique())
        if len(_timepoints) > 1:
            print("\nBY STUDY AND TIMEPOINT")
            display(
                long_df.groupby(["study", "timepoint"], dropna=False).agg(
                    rows=("standard_name", "size"),
                    participants=("participant_id", "nunique"),
                    variables=("standard_name", "nunique"),
                )
            )

    _coverage = pd.DataFrame(manifest.get("variable_coverage") or [])
    if not _coverage.empty:
        print("\nVARIABLE COVERAGE")
        print(
            _coverage.groupby("status").size().rename("study-variable-timepoints")
            .to_string()
        )
        _attention = _coverage[~_coverage["status"].isin(["produced", "planned"])]
        if not _attention.empty:
            display(_attention[[
                "study", "standard_name", "timepoint", "status", "detail"
            ]].head(30))

    _derivations = pd.DataFrame(manifest.get("cross_dataset_derivations") or [])
    if not _derivations.empty:
        print("\nCROSS-DATASET DERIVATIONS")
        _derivation_columns = [
            name for name in [
                "study", "standard_name", "timepoint", "derived",
                "measured_retained", "missing_height", "missing_weight",
                "ambiguous_height", "ambiguous_weight",
                "material_disagreements",
            ]
            if name in _derivations.columns
        ]
        display(_derivations[_derivation_columns])

    _missing = manifest.get("missing_variables") or []
    print(f"\nMISSING REQUESTED VARIABLES OR VALUES: {len(_missing)}")
    for _record in _missing[:20]:
        _location = _record.get("study")
        if _record.get("dataset_id"):
            _location += f"/{_record['dataset_id']}"
        print(f"  {_location}: {_record.get('standard_name')} — "
              f"{_record.get('reason')}")

    _drift = [
        _d for _d in (manifest.get("categorical_recode_diagnostics") or [])
        if _d.get("unexpected_value_examples")
    ]
    print(f"\nCATEGORICAL RECODE CHECK: {len(_drift)} unmapped source-code group(s)")
    if not _drift:
        print("  ✓ Every observed categorical source code was covered by its mapping.")
    for _d in _drift[:20]:
        _examples = ", ".join(str(_e.get("value")) for _e in _d["unexpected_value_examples"])
        print(f"  {_d.get('study')}/{_d.get('standard_name')} "
              f"({_d.get('source_variable')}): {_examples}")

    print("\nFIRST 10 HARMONIZED VALUES")
    _preview = long_df.copy()
    _preview["visit"] = _preview["timepoint"].fillna("not assigned")
    display(_preview[[
        "participant_id", "study", "visit", "standard_name", "value", "units"
    ]].head(10))
    print(
        "Categorical labels are stored in value; canonical category codes, when "
        "defined, are stored in value_numeric. Full source provenance is in "
        "harmonized_lineage.parquet."
    )

    print("\n" + "=" * 70)
    print("NEXT STEP — Table One handoff (copy into Step 1, or leave RUN_DIR = 'latest'):")
    print(f'  RUN_DIR = "{RUN_DIR}"')
    print("=" * 70)
